# Fine-Tuning DeepFake Detector

Use this notebook to correct the model when it fails on specific images (e.g., the Jack Nicholson deepfake).

### Step 1: Prepare Data
Ensure you have placed your difficult images in the following folders:
- **FAKE**: `/home/slim/FakeNews/data/DeepFake/Retrain/FAKE` (Put the deepfakes here)
- **REAL**: `/home/slim/FakeNews/data/DeepFake/Retrain/REAL` (Put valid real faces here to balance)

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam

# Paths
BASE_DIR = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(''))))
OLD_MODEL_PATH = '/home/slim/FakeNews/models/Final_DeepFake_Detector_85acc.h5'
NEW_MODEL_PATH = '/home/slim/FakeNews/models/DeepFake_Detector_v2.h5'
RETRAIN_DIR = '/home/slim/FakeNews/data/DeepFake/Retrain'

IMG_SIZE = 224
BATCH_SIZE = 4 # Small batch size for small retraining sets

In [ ]:
# 1. Load the existing model
if not os.path.exists(OLD_MODEL_PATH):
    raise FileNotFoundError(f"Model not found at {OLD_MODEL_PATH}")

print(f"Loading model from {OLD_MODEL_PATH}...")
model = load_model(OLD_MODEL_PATH)
print("Model loaded successfully.")

In [ ]:
# 2. Prepare Data Generator
print(f"Reading data from {RETRAIN_DIR}...")

datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    horizontal_flip=True,
    rotation_range=20,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2], # Augment brightness to help with lighting diffs
    fill_mode='nearest'
)

train_generator = datagen.flow_from_directory(
    RETRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True
)

# Check if we actually found files
if train_generator.samples == 0:
    print("\n⚠️ WARNING: No images found! Please add images to:")
    print(f"- {os.path.join(RETRAIN_DIR, 'FAKE')}")
    print(f"- {os.path.join(RETRAIN_DIR, 'REAL')}")
else:
    print(f"\nFound {train_generator.samples} images for retraining.")

In [ ]:
# 3. Configure for Fine-Tuning

# Ensure the model is trainable
model.trainable = True

# Use a very low learning rate to nudge the weights without forgetting everything
optimizer = Adam(learning_rate=1e-5) 

model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Model compiled with low learning rate (1e-5).")

In [ ]:
# 4. Train
if train_generator.samples > 0:
    print("Starting fine-tuning...")
    # Train for a few epochs (increase if you have many images)
    model.fit(
        train_generator,
        epochs=10, 
        steps_per_epoch=max(1, len(train_generator))
    )
    
    # 5. Save New Model
    model.save(NEW_MODEL_PATH)
    print(f"\n✅ Fine-tuned model saved to: {NEW_MODEL_PATH}")
    print("You can now change the backend to point to this new model file.")
else:
    print("Skipping training because no data was found.")